In [6]:
!pip install pykeops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.3/100.3 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.3/243.3 kB 17.2 MB/s eta 0:00:00
  Created wheel for pykeops: filename=pykeops-2.2.3-py3-none-any.whl size=118636 sha256=9ab127395cdd6ee0295d7e924e82112b3163b61ac64191c1b1b436d42c38d497
  Stored in directory: /root/.cache/pip/wheels/87/7c/b7/bf0e0c414ec6a9cb57d4f638c98e651f4cc115d3bdcecd1bff
  Created wheel for keopscore: filename=keopscore-2.2.3-py3-none-any.whl size=172483 sha256=24a279c500fd32beb195217c288a1179cdc9c40525f806d0591dc3ce36e71edb
  Stored in directory: /root/.cache/pip/wheels/4f/30/66/e08e82125e978761a47296a42998557febc34b08c52b5c8f88
Successfully built pykeops keopscore


In [1]:
import jax
import jax.numpy as jnp
from jax import lax, device_put
from functools import partial
import numpy as np

# --- Jitted Helper Functions ---

@jax.jit
def _compute_all_projections(data, proj_dirs):
    projs = data @ proj_dirs.T
    sorted_idxs = jnp.argsort(projs, axis=0)
    sorted_projs = jnp.take_along_axis(projs, sorted_idxs, axis=0)
    return sorted_projs.T, sorted_idxs.T

@partial(jax.jit, static_argnums=(4,5))
def _candidate_for_direction_jax(q, sorted_proj, sorted_idx, proj_dir, window_size, n):
    q_val = jnp.dot(q, proj_dir)
    idx = jnp.searchsorted(sorted_proj, q_val)
    slice_size = 2 * window_size
    start = jnp.clip(idx - window_size, 0, n - slice_size)
    return lax.dynamic_slice(sorted_idx, (start,), (slice_size,)).reshape(-1)

@partial(jax.jit, static_argnums=(4,5))
def _compute_candidate_matrix(q, sorted_projs, sorted_idxs, proj_dirs, window_size, n):
    return jax.vmap(
        _candidate_for_direction_jax,
        in_axes=(None, 0, 0, 0, None, None)
    )(q, sorted_projs, sorted_idxs, proj_dirs, window_size, n)

@jax.jit
def _compute_dists(q, candidate_points):
    return jnp.sum((candidate_points - q)**2, axis=1)

# --- Optimized AMPIIndex Class with Device Placement ---

class AMPIIndex:
    def __init__(self, data, num_projections):
        # Explicitly place data on TPU
        self.data = jnp.array(data, dtype=jnp.float32)
        self.data = device_put(self.data)
        self.n, self.d = self.data.shape
        self.L = num_projections

        key = jax.random.PRNGKey(0)
        raw_dirs = jax.random.normal(key, (self.L, self.d), dtype=jnp.float32)
        raw_dirs = device_put(raw_dirs)
        norms = jnp.linalg.norm(raw_dirs, axis=1, keepdims=True)
        self.proj_dirs = raw_dirs / norms

        self.sorted_projs, self.sorted_idxs = _compute_all_projections(self.data, self.proj_dirs)

    def add_vector(self, v):
        v = device_put(jnp.array(v, dtype=jnp.float32))
        new_idx = self.n
        self.data = jnp.concatenate([self.data, v[None, :]], axis=0)
        self.n += 1

        new_projections = v @ self.proj_dirs.T
        # Use JIT-compiled insertion
        updated_projs = []
        updated_idxs = []
        for i in range(self.L):
            sorted_proj, sorted_idx = self.sorted_projs[i], self.sorted_idxs[i]
            pos = int(np.array(jnp.searchsorted(sorted_proj, new_projections[i])))
            sorted_proj = jnp.concatenate([sorted_proj[:pos], new_projections[i:i+1], sorted_proj[pos:]])
            sorted_idx = jnp.concatenate([sorted_idx[:pos], jnp.array([new_idx]), sorted_idx[pos:]])
            updated_projs.append(sorted_proj)
            updated_idxs.append(sorted_idx)

        self.sorted_projs = jnp.stack(updated_projs)
        self.sorted_idxs = jnp.stack(updated_idxs)

    def remove_vector(self, idx):
        self.data = jnp.delete(self.data, idx, axis=0)
        self.n -= 1

        updated_projs = []
        updated_idxs = []
        for i in range(self.L):
            sorted_proj, sorted_idx = self.sorted_projs[i], self.sorted_idxs[i]
            pos = int(np.array(jnp.argmax(sorted_idx == idx)))
            sorted_proj = jnp.concatenate([sorted_proj[:pos], sorted_proj[pos+1:]])
            sorted_idx = jnp.concatenate([sorted_idx[:pos], sorted_idx[pos+1:]])
            sorted_idx = jnp.where(sorted_idx > idx, sorted_idx - 1, sorted_idx)
            updated_projs.append(sorted_proj)
            updated_idxs.append(sorted_idx)

        self.sorted_projs = jnp.stack(updated_projs)
        self.sorted_idxs = jnp.stack(updated_idxs)

    def query_candidates(self, q, window_size=32, max_candidates=1024):
        candidates = _compute_candidate_matrix(q, self.sorted_projs, self.sorted_idxs,
                                              self.proj_dirs, window_size, self.n).reshape(-1)
        # Use jnp.unique with fixed output size
        unique_candidates = jnp.unique(candidates, size=max_candidates, fill_value=-1)
        return unique_candidates  # shape: (max_candidates,)

    def query(self, q, window_size=32, k=1, max_candidates=1024):
        candidates = self.query_candidates(q, window_size, max_candidates)
        # No filtering here; defer filtering
        candidate_points = self.data[candidates]  # candidates include padding -1 indices
        mask = candidates >= 0
        # replace invalid candidates with first data point to avoid indexing errors
        safe_candidates = jnp.where(candidates >= 0, candidates, 0)
        candidate_points = self.data[safe_candidates]

        dists = _compute_dists(q, candidate_points)
        # mask invalid entries
        dists = jnp.where(candidates >= 0, dists, jnp.inf)
        topk_idx = jnp.argsort(dists)[:k]
        return candidate_points[topk_idx], dists[topk_idx], candidates[topk_idx]


In [17]:
# --- Tests with Printouts ---

import gc

def test_initialization():
    print("Running test_initialization...")
    data = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
    index = AMPIIndex(data, num_projections=3)
    print("Data:\n", np.array(index.data))
    print("Sorted indices:\n", np.array(index.sorted_idxs))
    print("Sorted projections:\n", np.array(index.sorted_projs))
    assert index.data.shape[0] == 3, "Data shape mismatch on initialization"
    assert index.n == 3, "n should be 3 on initialization"
    assert index.sorted_projs.shape == (3, 3), "sorted_projs shape mismatch"
    assert index.sorted_idxs.shape == (3, 3), "sorted_idxs shape mismatch"
    del index
    gc.collect()
    print("test_initialization passed.\n")

def test_query():
    print("Running test_query...")
    data = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
    index = AMPIIndex(data, num_projections=3)
    query_point = np.array([3, 4], dtype=np.float32)
    points, dists, indices = index.query(query_point, window_size=1, k=1)
    print("Query point:", query_point)
    print("Returned neighbor points:\n", np.array(points))
    print("Distances:\n", np.array(dists))
    print("Candidate indices:\n", np.array(indices))
    assert points.shape[0] == 1, "Query did not return one neighbor"
    assert dists.shape[0] == 1, "Distance vector not of length 1"
    assert indices.shape[0] == 1, "Candidate indices vector not of length 1"
    del index
    gc.collect()
    print("test_query passed.\n")

def test_add_vector():
    print("Running test_add_vector...")
    data = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
    index = AMPIIndex(data, num_projections=3)
    prev_n = index.n
    new_vector = np.array([7, 8], dtype=np.float32)
    index.add_vector(new_vector)
    print("Data after addition:\n", np.array(index.data))
    print("Sorted indices after addition:\n", np.array(index.sorted_idxs))
    print("Sorted projections after addition:\n", np.array(index.sorted_projs))
    assert index.n == prev_n + 1, "n did not increment after add"
    assert index.data.shape[0] == prev_n + 1, "Data shape not updated after add"
    assert index.sorted_projs.shape == (index.L, index.n), "sorted_projs shape mismatch after add"
    assert index.sorted_idxs.shape == (index.L, index.n), "sorted_idxs shape mismatch after add"
    del index
    gc.collect()
    print("test_add_vector passed.\n")

def test_remove_vector():
    print("Running test_remove_vector...")
    data = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
    index = AMPIIndex(data, num_projections=3)
    prev_n = index.n
    index.remove_vector(1)
    print("Data after removal:\n", np.array(index.data))
    print("Sorted indices after removal:\n", np.array(index.sorted_idxs))
    print("Sorted projections after removal:\n", np.array(index.sorted_projs))
    assert index.n == prev_n - 1, "n did not decrement after remove"
    assert index.data.shape[0] == prev_n - 1, "Data shape not updated after remove"
    assert index.sorted_projs.shape == (index.L, index.n), "sorted_projs shape mismatch after remove"
    assert index.sorted_idxs.shape == (index.L, index.n), "sorted_idxs shape mismatch after remove"
    del index
    gc.collect()
    print("test_remove_vector passed.\n")

def test_query_after_updates():
    print("Running test_query_after_updates...")
    data = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
    index = AMPIIndex(data, num_projections=3)
    new_vector = np.array([7, 8], dtype=np.float32)
    index.add_vector(new_vector)
    index.remove_vector(0)  # remove first vector
    query_point = np.array([7, 8], dtype=np.float32)
    points, dists, indices = index.query(query_point, window_size=1, k=1)
    print("Query point after updates:", query_point)
    print("Returned neighbor points:\n", np.array(points))
    print("Distances:\n", np.array(dists))
    print("Candidate indices:\n", np.array(indices))
    assert points.shape[0] == 1, "Query after updates did not return one neighbor"
    assert dists.shape[0] == 1, "Distance vector after updates not of length 1"
    assert indices.shape[0] == 1, "Candidate indices vector after updates not of length 1"
    del index
    gc.collect()
    print("test_query_after_updates passed.\n")

def test_add_remove_consistency():
    print("Running test_add_remove_consistency...")
    # Create a simple dataset.
    data = np.array([[1, 2],
                     [3, 4],
                     [5, 6]], dtype=np.float32)
    index = AMPIIndex(data, num_projections=3)

    # Save initial sorted indices and projections.
    initial_sorted_idxs = np.array(index.sorted_idxs)
    initial_sorted_projs = np.array(index.sorted_projs)

    print("Initial sorted indices:\n", initial_sorted_idxs)
    print("Initial sorted projections:\n", initial_sorted_projs)

    # Add a new vector.
    new_vector = np.array([7, 8], dtype=np.float32)
    index.add_vector(new_vector)
    print("After addition, sorted indices:\n", np.array(index.sorted_idxs))
    print("After addition, sorted projections:\n", np.array(index.sorted_projs))

    # Remove the newly added vector (should be at index n-1).
    index.remove_vector(index.n - 1)

    # Get final sorted indices and projections.
    final_sorted_idxs = np.array(index.sorted_idxs)
    final_sorted_projs = np.array(index.sorted_projs)

    print("Final sorted indices:\n", final_sorted_idxs)
    print("Final sorted projections:\n", final_sorted_projs)

    assert np.array_equal(initial_sorted_idxs, final_sorted_idxs), "Sorted indices mismatch after add/remove"
    assert np.allclose(initial_sorted_projs, final_sorted_projs), "Sorted projections mismatch after add/remove"
    del index
    gc.collect()
    print("test_add_remove_consistency passed.\n")

def test_random_insertion_removal_consistency():
    print("Running test_random_insertion_removal_consistency...")
    # Create a base dataset.
    data = np.array([[1, 2],
                     [3, 4],
                     [5, 6]], dtype=np.float32)
    index = AMPIIndex(data, num_projections=3)
    base_n = index.n
    initial_sorted_idxs = np.array(index.sorted_idxs)
    initial_sorted_projs = np.array(index.sorted_projs)
    print("Initial sorted indices:\n", initial_sorted_idxs)
    print("Initial sorted projections:\n", initial_sorted_projs)

    # Randomly decide to add between 3 and 7 new vectors.
    num_new = np.random.randint(3, 8)
    print(f"Adding {num_new} random vectors.")
    # Add random vectors.
    for _ in range(num_new):
        new_vector = np.random.rand(index.d).astype(np.float32) * 10
        index.add_vector(new_vector)
    print("After additions, data:\n", np.array(index.data))
    print("After additions, sorted indices:\n", np.array(index.sorted_idxs))

    # Now remove the newly added vectors in random order.
    # The new vectors are always at indices base_n to index.n-1.
    current_new_indices = list(range(base_n, index.n))
    np.random.shuffle(current_new_indices)
    print("Random removal order (indices from new block):", current_new_indices)
    for _ in range(len(current_new_indices)):
        # Always remove one vector from the new block (which are at indices from base_n to current n-1).
        current_new_block = list(range(base_n, index.n))
        # Choose a random index from the new block.
        remove_idx = np.random.choice(current_new_block)
        index.remove_vector(remove_idx)
        print(f"Removed vector at index {remove_idx}. Current n: {index.n}")

    final_sorted_idxs = np.array(index.sorted_idxs)
    final_sorted_projs = np.array(index.sorted_projs)
    print("Final sorted indices after random removals:\n", final_sorted_idxs)
    print("Final sorted projections after random removals:\n", final_sorted_projs)

    # The final state should match the original base dataset.
    assert np.array_equal(initial_sorted_idxs, final_sorted_idxs), "Random add/remove: Sorted indices mismatch"
    assert np.allclose(initial_sorted_projs, final_sorted_projs), "Random add/remove: Sorted projections mismatch"
    del index
    gc.collect()
    print("test_random_insertion_removal_consistency passed.\n")

def run_tests():
    print("Running sanity tests ...\n")
    test_initialization()
    test_query()
    test_add_vector()
    test_remove_vector()
    test_query_after_updates()
    test_add_remove_consistency()
    test_random_insertion_removal_consistency()
    print("All tests passed successfully.")

"""
Running sanity tests for AMPI ...
"""
run_tests()

Running sanity tests ...

Running test_initialization...
Data:
 [[1. 2.]
 [3. 4.]
 [5. 6.]]
Sorted indices:
 [[0 1 2]
 [2 1 0]
 [2 1 0]]
Sorted projections:
 [[ 2.1860895  4.997445   7.8088007]
 [-5.990224  -3.6654968 -1.34077  ]
 [-5.012686  -3.4012058 -1.7897259]]
test_initialization passed.

Running test_query...
Query point: [3. 4.]
Returned neighbor points:
 [[3. 4.]]
Distances:
 [0.]
Candidate indices:
 [1]
test_query passed.

Running test_add_vector...
Data after addition:
 [[1. 2.]
 [3. 4.]
 [5. 6.]
 [7. 8.]]
Sorted indices after addition:
 [[0 1 2 3]
 [3 2 1 0]
 [3 2 1 0]]
Sorted projections after addition:
 [[ 2.1860895  4.997445   7.8088007 10.620155 ]
 [-8.314951  -5.990224  -3.6654968 -1.34077  ]
 [-6.6241655 -5.012686  -3.4012058 -1.7897259]]
test_add_vector passed.

Running test_remove_vector...
Data after removal:
 [[1. 2.]
 [5. 6.]]
Sorted indices after removal:
 [[0 1]
 [1 0]
 [1 0]]
Sorted projections after removal:
 [[ 2.1860895  7.8088007]
 [-5.990224  -1.34077  ]


In [13]:
import jax
import jax.numpy as jnp
import numpy as np
from torchvision import datasets
from torchvision.datasets import MNIST
import gc

# Assume AMPIIndex is your optimized JAX version already implemented.

# Load dataset (do once, CPU side)
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

# Flatten images and normalize pixel values.
train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
mnist_data = np.concatenate([train_data, test_data], axis=0)
N, D = mnist_data.shape
print(f"MNIST loaded: {N} points, dimension {D}")

# Create AMPI index on TPU
ampi = AMPIIndex(mnist_data, num_projections=128)

# JIT compile the query function outside the loop once
@partial(jax.jit, static_argnums=(1,2,3))
def batched_query(queries, window_size=32, k=1, max_candidates=1024):
    def single_query(q):
        return ampi.query(q, window_size, k, max_candidates)
    return jax.vmap(single_query, in_axes=0)(queries)

# Prepare batch of queries:
query_points = mnist_data[:1000]  # For example, 100 queries at once
window_size = 128
k = 50

# Run batched queries on TPU
import time
start = time.perf_counter()
results_points, results_dists, results_indices = batched_query(query_points, window_size, k)
jax.block_until_ready(results_points)  # Ensure synchronization
end = time.perf_counter()
print(f"Batched query took {end - start:.2f} seconds.")
del ampi
gc.collect()

MNIST loaded: 70000 points, dimension 784
Batched query took 10.92 seconds.


0

In [7]:
import torch
import numpy as np
from torchvision.datasets import MNIST
from pykeops.torch import LazyTensor
import time
import gc

def batched_knn_pykeops(data, queries, k=1):
    X_i = LazyTensor(queries[:, None, :])   # Shape: (num_queries, 1, D)
    X_j = LazyTensor(data[None, :, :])      # Shape: (1, N, D)
    D_ij = ((X_i - X_j) ** 2).sum(-1)       # Shape: (num_queries, N)
    indices = D_ij.argKmin(k, dim=1)        # Shape: (num_queries, k)
    return indices

# --- Load MNIST data ---
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
mnist_data = torch.cat([train_data, test_data], dim=0).contiguous()
mnist_np = mnist_data.numpy()
N, D = mnist_data.shape
print(f"MNIST loaded: {N} points, dimension {D}")

# --- Prepare queries ---
query_points = mnist_data[:1000]
k=50

# --- Perform batched k-NN with PyKeOps ---
start = time.perf_counter()
indices_pykeops = batched_knn_pykeops(mnist_data, query_points, k)
indices_pykeops = indices_pykeops.cpu().numpy()
end = time.perf_counter()

print(f"PyKeOps batched query (k={k}) took {end - start:.2f} seconds.")

[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... OK
MNIST loaded: 70000 points, dimension 784
[KeOps] Generating code for ArgKMin_Reduction reduction (with parameters 0) of formula Sum((a-b)**2) with a=Var(0,784,0), b=Var(1,784,1) ... OK
PyKeOps batched query (k=50) took 1.27 seconds.


In [1]:
import jax
import jax.numpy as jnp
from functools import partial

@partial(jax.jit, static_argnums=(2, 3))
def k_nearest_neighbors_tpu(x, y, k=5, batch_size=1024):
    n_queries = y.shape[0]
    num_batches = (n_queries + batch_size - 1) // batch_size

    @partial(jax.jit, static_argnums=(2,))
    def batch_knn(y_batch, x, k):
        distances = jnp.sum((y_batch[:, None, :] - x[None, :, :]) ** 2, axis=-1)
        topk_neg_distances, topk_indices = jax.lax.top_k(-distances, k)
        return topk_indices  # These are global indices w.r.t. x

    results = []
    for i in range(num_batches):
        start = i * batch_size
        end = min((i + 1) * batch_size, n_queries)
        y_batch = y[start:end]
        knn_global_indices = batch_knn(y_batch, x, k)  # These are already global w.r.t. x
        results.append(knn_global_indices)

    knn_indices = jnp.concatenate(results, axis=0)
    return knn_indices

# Example usage:
if __name__ == "__main__":
    key = jax.random.PRNGKey(0)
    x_data = jax.random.normal(key, (60000, 784))  # Database (real data)
    y_data = jax.random.normal(key, (10000, 784))  # Queries

    import time
    t0 = time.time()
    knn_indices = k_nearest_neighbors_tpu(x_data, y_data, k=5, batch_size=4096).block_until_ready()
    t1 = time.time()

    print(f"k-NN computed in {t1 - t0:.3f} seconds.")
    print("Shape of k-NN indices:", knn_indices.shape)


k-NN computed in 4.376 seconds.
Shape of k-NN indices: (10000, 5)


In [2]:
knn_indices

Array([[    0, 58896, 13000, 29000, 38131],
       [    1,  1389, 51560, 41355,  4315],
       [    2, 24794, 35323,  2404, 20623],
       ...,
       [ 9997, 47537, 16764, 22384, 14154],
       [ 9998, 22088, 36916, 13327,  1034],
       [ 9999,  8273, 26302, 18958, 36144]], dtype=int32)

In [9]:
from torchvision import datasets
import numpy as np

# Example: Efficient MNIST-sized data (60,000 x 784)
# --- Load MNIST data ---
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
mnist_data = np.concatenate([train_data, test_data], axis=0)
N, D = mnist_data.shape
print(f"MNIST loaded: {N} points, dimension {D}")

# --- Prepare queries ---
query_points = mnist_data[:1000]
k=50

# Trigger compilation and execution on TPU
import time
start_time = time.time()
knn_indices = k_nearest_neighbors_tpu(mnist_data, query_points, k, batch_size=4096).block_until_ready()
end_time = time.time()

print("Nearest neighbors computed:", knn_indices.shape)
print("Time taken:", end_time - start_time, "seconds")

MNIST loaded: 70000 points, dimension 784
Nearest neighbors computed: (1000, 50)
Time taken: 0.9913358688354492 seconds


In [10]:
for i in range(1000):
    print('-'*50)
    print(knn_indices[i])
    print(indices_pykeops[i])

Streaming output truncated to the last 5000 lines.
 43647 19079]
[  545 52329 59838 52401 32127 46397 26028 52237  3211 32181 18795  7297
 24123 50524  5725 46121  5791 40629 34001 32105 43787 19605 51715 53285
 44507 40665 51705 21146 52711  2043 38742 18114 52303 64371 51725 20539
 53737  3891 69817 16145 35365 33813 32103 21105 51709 65065 69854 46395
 43647 19079]
--------------------------------------------------
[  546 66876 66821 35859 13704 34190 17496 38867  3386  1486 46350 25078
 51268 21056 24969 46072 31446 44192 45940 69156 55580 47872 47514  8048
 26781 62083 50372 52476 12108 17731 14810 53488 18675 58582 65892 20400
 13142 31960 62712 49560 57919 23572 42908 45822 66322  9004 42534  8957
 66918 45088]
[  546 66876 66821 35859 13704 34190 17496 38867  3386  1486 46350 25078
 51268 21056 24969 46072 31446 44192 45940 69156 55580 47872 47514  8048
 26781 62083 50372 52476 12108 17731 14810 53488 18675 58582 65892 20400
 13142 31960 62712 49560 57919 23572 42908 45822 6632

In [75]:
import jax
import jax.numpy as jnp
from functools import partial

@partial(jax.jit, static_argnums=(2, 3, 4))
def k_nearest_neighbors_tpu(x, y, k=5, batch_size_y=1024, batch_size_x=1024):
    n_queries = y.shape[0]
    n_database = x.shape[0]
    num_batches_y = (n_queries + batch_size_y - 1) // batch_size_y
    num_batches_x = (n_database + batch_size_x - 1) // batch_size_x

    @partial(jax.jit, static_argnums=(2,))
    def batch_knn(y_batch, x_batch, k, start_x):
        distances = jnp.sum((y_batch[:, None, :] - x_batch[None, :, :]) ** 2, axis=-1)
        topk_neg_distances, topk_local_indices = jax.lax.top_k(-distances, k)
        topk_global_indices = topk_local_indices + start_x
        return topk_neg_distances, topk_global_indices

    results = []
    for i in range(num_batches_y):
        start_y = i * batch_size_y
        end_y = min((i + 1) * batch_size_y, n_queries)
        y_batch = y[start_y:end_y]

        all_distances = []
        all_indices = []
        for j in range(num_batches_x):
            start_x = j * batch_size_x
            end_x = min((j + 1) * batch_size_x, n_database)
            x_batch = x[start_x:end_x]

            batch_distances, batch_global_indices = batch_knn(y_batch, x_batch, k, start_x)
            all_distances.append(batch_distances)
            all_indices.append(batch_global_indices)

        all_distances = jnp.concatenate(all_distances, axis=1)
        all_indices = jnp.concatenate(all_indices, axis=1)

        # Select global top-k from merged batches
        topk_neg_distances, topk_indices = jax.lax.top_k(all_distances, k)
        topk_global_indices = jnp.take_along_axis(all_indices, topk_indices, axis=1)

        results.append(topk_global_indices)

    knn_indices = jnp.concatenate(results, axis=0)
    return knn_indices

# Example usage:
if __name__ == "__main__":
    key = jax.random.PRNGKey(0)
    x_data = jax.random.normal(key, (60000, 784))  # Database (real data)
    y_data = jax.random.normal(key, (10000, 784))  # Queries

    import time
    t0 = time.time()
    knn_indices = k_nearest_neighbors_tpu(x_data, y_data, k=5, batch_size_y=4096, batch_size_x=10*8192).block_until_ready()
    t1 = time.time()

    print(f"k-NN computed in {t1 - t0:.3f} seconds.")
    print("Shape of k-NN indices:", knn_indices.shape)


k-NN computed in 4.425 seconds.
Shape of k-NN indices: (10000, 5)


In [76]:
from torchvision import datasets

# Example: Efficient MNIST-sized data (60,000 x 784)
# --- Load MNIST data ---
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
mnist_data = np.concatenate([train_data, test_data], axis=0)
N, D = mnist_data.shape
print(f"MNIST loaded: {N} points, dimension {D}")

# --- Prepare queries ---
query_points = mnist_data[:1000]
k=50

# Trigger compilation and execution on TPU
import time
start_time = time.time()
knn_indices = k_nearest_neighbors_tpu(mnist_data, query_points, k, batch_size_y=4096, batch_size_x=10*8192).block_until_ready()
end_time = time.time()

print("Nearest neighbors computed:", nn_indices.shape)
print("Time taken:", end_time - start_time, "seconds")

MNIST loaded: 70000 points, dimension 784
Nearest neighbors computed: (1000, 50)
Time taken: 2.0226337909698486 seconds


In [77]:
for i in range(1000):
    print('-'*50)
    print(knn_indices[i])
    print(indices_pykeops[i])

Streaming output truncated to the last 5000 lines.
 43647 19079]
[  545 52329 59838 52401 32127 46397 26028 52237  3211 32181 18795  7297
 24123 50524  5725 46121  5791 40629 34001 32105 43787 19605 51715 53285
 44507 40665 51705 21146 52711  2043 38742 18114 52303 64371 51725 20539
 53737  3891 69817 16145 35365 33813 32103 21105 51709 65065 69854 46395
 43647 19079]
--------------------------------------------------
[  546 66876 66821 35859 13704 34190 17496 38867  3386  1486 46350 25078
 51268 21056 24969 46072 31446 44192 45940 69156 55580 47872 47514  8048
 26781 62083 50372 52476 12108 17731 14810 53488 18675 58582 65892 20400
 13142 31960 62712 49560 57919 23572 42908 45822 66322  9004 42534  8957
 66918 45088]
[  546 66876 66821 35859 13704 34190 17496 38867  3386  1486 46350 25078
 51268 21056 24969 46072 31446 44192 45940 69156 55580 47872 47514  8048
 26781 62083 50372 52476 12108 17731 14810 53488 18675 58582 65892 20400
 13142 31960 62712 49560 57919 23572 42908 45822 6632

In [2]:
import jax
import jax.numpy as jnp
from jax import lax
from functools import partial
import numpy as np

# --- Module-Level Helper Functions for Updates (Not Jitted) ---

def insert_projection(sorted_proj, sorted_idx, new_val, new_idx):
    # Compute insertion position as a concrete Python int.
    pos = int(np.array(jnp.searchsorted(sorted_proj, new_val)))
    new_val_arr = jnp.array([new_val], dtype=sorted_proj.dtype)
    new_idx_arr = jnp.array([new_idx], dtype=sorted_idx.dtype)
    new_sorted_proj = jnp.concatenate((sorted_proj[:pos], new_val_arr, sorted_proj[pos:]))
    new_sorted_idx = jnp.concatenate((sorted_idx[:pos], new_idx_arr, sorted_idx[pos:]))
    return new_sorted_proj, new_sorted_idx

def remove_projection(sorted_proj, sorted_idx, remove_idx):
    # Find the position of the vector to remove as a concrete Python int.
    pos = int(np.array(jnp.argmax(sorted_idx == remove_idx)))
    new_sorted_proj = jnp.concatenate((sorted_proj[:pos], sorted_proj[pos+1:]))
    new_sorted_idx = jnp.concatenate((sorted_idx[:pos], sorted_idx[pos+1:]))
    new_sorted_idx = jnp.where(new_sorted_idx > remove_idx, new_sorted_idx - 1, new_sorted_idx)
    return new_sorted_proj, new_sorted_idx

# --- Jitted Functions for Querying ---
# We mark window_size and n as static to ensure concrete slice sizes.

@jax.jit
def compute_all_projections(data, proj_dirs):
    # data: (n, d), proj_dirs: (L, d)
    projs = data @ proj_dirs.T   # (n, L)
    sorted_idxs = jnp.argsort(projs, axis=0)  # (n, L)
    sorted_projs = jnp.take_along_axis(projs, sorted_idxs, axis=0)  # (n, L)
    # Transpose to shape (L, n)
    return sorted_projs.T, sorted_idxs.T

@partial(jax.jit, static_argnums=(4,5))
def candidate_for_direction_jax(q, sorted_proj, sorted_idx, proj_dir, window_size, n):
    # Compute projection of q onto proj_dir.
    q_val = jnp.dot(q, proj_dir)
    idx = jnp.searchsorted(sorted_proj, q_val)
    slice_size = 2 * window_size  # concrete because window_size is static
    start = jnp.clip(idx - window_size, 0, n - slice_size)
    # Use dynamic_slice with dynamic start but fixed slice size.
    sliced = lax.dynamic_slice(sorted_idx, (start,), (slice_size,))
    return sliced.reshape(-1)

@partial(jax.jit, static_argnums=(4,5))
def _compute_candidate_matrix(q, sorted_projs, sorted_idxs, proj_dirs, window_size, n):
    return jax.vmap(
        candidate_for_direction_jax,
        in_axes=(None, 0, 0, 0, None, None)
    )(q, sorted_projs, sorted_idxs, proj_dirs, window_size, n)

def query_candidates_jax(q, sorted_projs, sorted_idxs, proj_dirs, window_size, n):
    # Ensure window_size and n are Python ints.
    candidate_matrix = _compute_candidate_matrix(q, sorted_projs, sorted_idxs, proj_dirs, int(window_size), int(n))
    flat = candidate_matrix.reshape(-1)
    return jnp.unique(flat)

# --- AMPIIndex Class (Main Code; remains intact) ---

class AMPIIndex:
    def __init__(self, data, num_projections):
        """
        Initializes the AMPI index for TPU computation using JAX.

        Args:
          data: A NumPy array of shape (n, d) containing n data points.
          num_projections: The number L of random projection directions.
        """
        self.data = jnp.array(data, dtype=jnp.float32)
        self.n, self.d = self.data.shape
        self.L = num_projections

        key = jax.random.PRNGKey(0)
        raw_dirs = jax.random.normal(key, (self.L, self.d), dtype=jnp.float32)
        norms = jnp.linalg.norm(raw_dirs, axis=1, keepdims=True)
        self.proj_dirs = raw_dirs / norms

        self.sorted_projs, self.sorted_idxs = compute_all_projections(self.data, self.proj_dirs)

    def add_vector(self, v):
        """
        Adds a new vector to the index by computing its projections and
        inserting them into the pre-sorted projection arrays.

        Args:
          v: A vector of shape (d,) to be added to the index.
        """
        v = jnp.array(v, dtype=jnp.float32)
        new_idx = self.n  # new vector's index
        self.data = jnp.concatenate([self.data, v[None, :]], axis=0)
        self.n += 1

        new_projections = v @ self.proj_dirs.T  # shape (L,)
        updated_projs = []
        updated_idxs = []
        for i in range(self.L):
            sp, si = insert_projection(self.sorted_projs[i],
                                       self.sorted_idxs[i],
                                       new_projections[i],
                                       new_idx)
            updated_projs.append(sp)
            updated_idxs.append(si)
        self.sorted_projs = jnp.stack(updated_projs)
        self.sorted_idxs = jnp.stack(updated_idxs)

    def remove_vector(self, idx):
        """
        Removes a vector from the index by removing its projection and adjusting
        indices for all projection directions.

        Args:
          idx: Integer index of the vector to be removed.
        """
        self.data = jnp.delete(self.data, idx, axis=0)
        self.n -= 1

        updated_projs = []
        updated_idxs = []
        for i in range(self.L):
            sp, si = remove_projection(self.sorted_projs[i],
                                       self.sorted_idxs[i],
                                       idx)
            updated_projs.append(sp)
            updated_idxs.append(si)
        self.sorted_projs = jnp.stack(updated_projs)
        self.sorted_idxs = jnp.stack(updated_idxs)

    def query_candidates(self, q, window_size=10):
        """
        Extracts candidate indices for the query point q.

        Args:
          q: Query point of shape (d,).
          window_size: Half-window size for candidate extraction.

        Returns:
          A 1D JAX array of unique candidate indices.
        """
        q = jnp.array(q, dtype=jnp.float32)
        return query_candidates_jax(q, self.sorted_projs, self.sorted_idxs,
                                    self.proj_dirs, int(window_size), int(self.n))

    def query(self, q, window_size=10, k=1):
        """
        Computes the k-NN result by:
          1. Extracting candidate indices.
          2. Computing full squared L2 distances.
          3. Returning the top k neighbors.

        Returns:
          final_points: (k, d) array of nearest neighbors.
          final_dists: (k,) array of squared distances.
          final_indices: Corresponding candidate indices.
        """
        q = jnp.array(q, dtype=jnp.float32)
        candidates = self.query_candidates(q, window_size)
        candidate_points = self.data[candidates]
        dists = jnp.sum((candidate_points - q)**2, axis=1)
        topk_idx = jnp.argsort(dists)[:k]
        return candidate_points[topk_idx], dists[topk_idx], candidates[topk_idx]